# 10 — SmolVLA batch training

Fine-tune the base checkpoint on 16 approved human-derived robot episodes. This is a fixed-scene proof of concept, not a generalization benchmark. Start with 2,000 updates, evaluate, then decide whether to continue. No simulator is needed during training. Use a GPU runtime and the working LeRobot + SmolVLA dependencies from notebook 07. Run top to bottom once; rerunning the model cell creates a fresh model. Periodic checkpoints include optimizer state for later continuation.

In [ ]:
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e ".[libero]"

In [ ]:
!pip install num2words

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import zipfile
import os

project = Path("/content/drive/MyDrive/Projects/humanoid-challenge")
SOURCE_RUN = "20261015T103228_810691Z"
archive_path = project / "data/lerobot_exports" / SOURCE_RUN / "lerobot_dataset.zip"
dataset_path = Path("/content/humanoid_dataset_exports") / SOURCE_RUN / "lerobot_dataset"

assert archive_path.is_file(), f"Archive missing: {archive_path}"

if not (dataset_path / "meta/info.json").is_file():
    dataset_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(dataset_path)

assert (dataset_path / "meta/info.json").is_file()
assert (dataset_path / "data").is_dir()

print("Dataset ready:", dataset_path)

In [ ]:
import json, random, time, shutil, hashlib
import numpy as np
import torch
from datetime import datetime, timezone
from torch.utils.data import DataLoader
from lerobot.datasets.lerobot_dataset import LeRobotDataset
assert torch.cuda.is_available(), 'Select a GPU runtime before loading the model.'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
REPO_ID = 'local/humanoid_human_batch'
train_set = LeRobotDataset(repo_id=REPO_ID, root=dataset_path)
assert len(train_set)==71274 and train_set.meta.total_episodes==16
assert train_set.meta.fps==20
conversion = json.loads((dataset_path/'conversion_manifest.json').read_text())
assert conversion['readback_verified'] and conversion['episode_boundary_padding_verified']
print('GPU:',torch.cuda.get_device_name(0))
print('Training frames:',len(train_set),'Episodes:',train_set.meta.total_episodes)


In [ ]:
optimizer = None
trainable_parameters = []
import gc
import torch

from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

assert torch.cuda.is_available(), "Enable a GPU runtime in Colab first."

# Release the previous model before loading another copy.
if "policy" in globals():
    del policy

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "lerobot/smolvla_base"

config = SmolVLAConfig.from_pretrained(MODEL_ID)
config.device = "cuda"

config.input_features = {
    "observation.images.image": PolicyFeature(
        type=FeatureType.VISUAL,
        shape=(3, 256, 256),
    ),
    "observation.images.image2": PolicyFeature(
        type=FeatureType.VISUAL,
        shape=(3, 256, 256),
    ),
    "observation.state": PolicyFeature(
        type=FeatureType.STATE,
        shape=(8,),
    ),
}

config.output_features = {
    "action": PolicyFeature(
        type=FeatureType.ACTION,
        shape=(7,),
    ),
}

config.chunk_size = 50
config.n_action_steps = 10
config.empty_cameras = 0

# Preserve the checkpoint's internal padded dimensions.
assert config.max_state_dim >= 8
assert config.max_action_dim >= 7

policy = SmolVLAPolicy.from_pretrained(
    MODEL_ID,
    config=config,
    strict=True,
).to("cuda")

print("Inputs:", policy.config.input_features)
print("Outputs:", policy.config.output_features)

In [ ]:
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    pretrained_path=None,
    dataset_stats=train_set.meta.stats,
)

In [ ]:
from torch.utils.data import DataLoader
from lerobot.datasets.lerobot_dataset import LeRobotDataset

training_dataset = LeRobotDataset(
    repo_id=REPO_ID,
    root=dataset_path,
    delta_timestamps={
        "action": [
            i / train_set.meta.fps
            for i in range(policy.config.chunk_size)
        ]
    },
)

loader = DataLoader(
    training_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

raw_batch = next(iter(loader))
model_batch = preprocess(raw_batch)

for key in [
    "observation.images.image",
    "observation.images.image2",
    "observation.state",
    "action",
]:
    value = model_batch[key]
    print(key, tuple(value.shape), value.device)
    assert torch.isfinite(value).all()

assert model_batch["observation.state"].shape == (1, 8)
assert model_batch["action"].shape == (1, 50, 7)

## Training and checkpoint configuration
Batch size 1 preserves the previously working memory footprint. A step samples one 50-action chunk; 2,000 updates are an initial budget, not a claim of convergence. All 16 episodes are training inputs. The checkpoint saves unnormalized action interpretation and dataset-specific processors. Do not multiply actions or change normalization for this run.

In [ ]:
TOTAL_STEPS = 2000
SAVE_EVERY = 500
LEARNING_RATE = 1e-5
trainable_parameters = [p for p in policy.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters,lr=LEARNING_RATE,weight_decay=1e-2)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
local_run = Path('/content/humanoid_training')/run_id
local_run.mkdir(parents=True,exist_ok=False)
drive_run = project/'outputs/smolvla_batch'/run_id
drive_run.mkdir(parents=True,exist_ok=False)
training_details = []
completed_steps = 0
last_saved_step = -1

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def save_checkpoint(step):
    global last_saved_step, checkpoint_dir
    checkpoint_dir=local_run/f'step_{step:06d}'
    checkpoint_dir.mkdir(parents=True,exist_ok=False)
    policy.save_pretrained(checkpoint_dir)
    preprocess.save_pretrained(checkpoint_dir,config_filename='policy_preprocessor.json')
    postprocess.save_pretrained(checkpoint_dir,config_filename='policy_postprocessor.json')
    torch.save({'step':step,'optimizer':optimizer.state_dict(),
                'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
                'numpy_rng':np.random.get_state(),'python_rng':random.getstate()},
               checkpoint_dir/'training_state.pt')
    # RNG state alone does not preserve an in-flight DataLoader iterator's order.
    log=dict(completed_steps=step,batch_size=loader.batch_size,learning_rate=LEARNING_RATE,
             seed=SEED,base_model=MODEL_ID,repo_id=REPO_ID,dataset_export=SOURCE_RUN,
             dataset_episodes=16,dataset_frames=71274,
             loss_history=[x['loss'] for x in training_details],
             purpose='fixed-scene multi-demonstration proof of concept; no held-out split')
    (checkpoint_dir/'training_log.json').write_text(json.dumps(log,indent=2))
    (checkpoint_dir/'training_details.json').write_text(json.dumps(training_details,indent=2))
    shutil.copy2(dataset_path/'conversion_manifest.json',checkpoint_dir/'dataset_conversion_manifest.json')
    archive=Path(shutil.make_archive(str(checkpoint_dir),'zip',root_dir=checkpoint_dir))
    destination=drive_run/archive.name
    temporary=destination.with_suffix('.zip.partial')
    print('Saving checkpoint to Drive:',destination,flush=True)
    shutil.copy2(archive,temporary)
    assert digest(temporary)==digest(archive), 'Checkpoint copy checksum mismatch'
    temporary.replace(destination)
    (drive_run/'latest.json').write_text(json.dumps({'step':step,'archive':destination.name,'sha256':digest(archive)},indent=2))
    last_saved_step=step
    print('Verified checkpoint:',destination,flush=True)


In [ ]:
assert completed_steps == 0, 'This cell already ran; do not silently restart its step counter.'
policy.train()
policy.reset()
iterator=iter(loader)
started=time.monotonic()
try:
    for step in range(1,TOTAL_STEPS+1):
        try: raw_batch=next(iterator)
        except StopIteration:
            iterator=iter(loader)
            raw_batch=next(iterator)
        model_batch=preprocess(raw_batch)
        optimizer.zero_grad(set_to_none=True)
        loss,metrics=policy.forward(model_batch)
        assert torch.isfinite(loss).item(), 'Nonfinite training loss'
        loss.backward()
        norm=torch.nn.utils.clip_grad_norm_(trainable_parameters,1.0,error_if_nonfinite=True)
        optimizer.step()
        completed_steps=step
        training_details.append(dict(step=step,loss=float(loss.detach().cpu()),
            gradient_norm_before_clipping=float(norm),
            frame_indices=raw_batch['index'].tolist(),
            episode_indices=raw_batch['episode_index'].tolist()))
        if step%20==0:
            recent=np.mean([x['loss'] for x in training_details[-20:]])
            print(f'Step {step}/{TOTAL_STEPS} | mean loss {recent:.4f} | elapsed {(time.monotonic()-started)/60:.1f} min',flush=True)
        if step%SAVE_EVERY==0: save_checkpoint(step)

except KeyboardInterrupt:
    print('Interrupted. Saving the last completed update if needed.',flush=True)
    if completed_steps and last_saved_step!=completed_steps:
        save_checkpoint(completed_steps)
    raise
if completed_steps and last_saved_step!=completed_steps:
    save_checkpoint(completed_steps)
print('Local checkpoint:',checkpoint_dir)
print('Drive checkpoints:',drive_run)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
history=pd.DataFrame(training_details)
ax=history.plot(x='step',y='loss',alpha=.3,figsize=(10,4),label='Per update')
ax.plot(history['step'],history['loss'].rolling(50,min_periods=1).mean(),label='50-step mean')
ax.set(yscale='log',ylabel='Training loss');ax.legend();plt.show()
print('Sampled chunks per episode:')
print(history['episode_indices'].explode().value_counts().sort_index())
print('Loss measures prediction on training data, not closed-loop task success.')


## Next: evaluate the saved batch checkpoint
Use this checkpoint and its saved processors for simulation evaluation. Do not reload the old single-episode smoke-test checkpoint. The local checkpoint directory is printed above; after a runtime restart, restore a verified step ZIP from Drive. Checkpoint archives include optimizer state but this notebook starts a fresh run; continuation must explicitly restore it. A hard Colab disconnect can lose updates since the last completed Drive checkpoint. Notebook 07 contains pilot-specific reset/diagnostic paths, so it needs adaptation before evaluating this batch model.

# Evaluate in this same runtime
Run only the cells below after training. They reuse the local checkpoint and allocated GPU; do not rerun training/setup above. Optimizer references are released to free GPU memory before loading the saved model for evaluation.


In [ ]:
import os
os.environ['MUJOCO_GL']='egl'
os.environ['PYOPENGL_PLATFORM']='egl'
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, zipfile, hashlib, subprocess, sys, time
import numpy as np
import torch
import h5py
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
project=Path('/content/drive/MyDrive/Projects/humanoid-challenge')
assert 'checkpoint_dir' in globals(), 'Run the training cells first in this session.'
checkpoint_dir = Path(checkpoint_dir)
training_run = checkpoint_dir.parent.name
assert (checkpoint_dir/'model.safetensors').is_file()
training_log=json.loads((checkpoint_dir/'training_log.json').read_text())
evaluated_step = int(training_log['completed_steps'])
assert evaluated_step > 0
assert training_log['repo_id']=='local/humanoid_human_batch'
conversion=json.loads((checkpoint_dir/'dataset_conversion_manifest.json').read_text())
source=conversion['sources'][0]
source_episode=project/source['source_h5']
print('Checkpoint:',checkpoint_dir)
print('Recorded scene:',source['demo'],source_episode)
# Initialize first-use LIBERO configuration in a process that can answer its prompt.
subprocess.run([sys.executable,'-c','import libero.libero'],input='n\n',text=True,check=True)


In [ ]:
import gc
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

# Drop optimizer references to the previous model before evaluation reload.
optimizer = None
trainable_parameters = []
if "policy" in globals():
    del policy
gc.collect()
torch.cuda.empty_cache()

policy = SmolVLAPolicy.from_pretrained(checkpoint_dir).to("cuda").eval()

preprocess, postprocess = make_pre_post_processors(
    policy.config,
    pretrained_path=checkpoint_dir,
    preprocessor_overrides={
        "device_processor": {"device": "cuda"},
    },
)

print(policy.config.output_features)
# Expect action shape=(7,)

In [ ]:
import numpy as np
import torch
from scipy.spatial.transform import Rotation

def make_policy_observation(obs, instruction):
    # Quaternion from LIBERO uses XYZW ordering.
    rotation_vector = Rotation.from_quat(obs["robot0_eef_quat"]).as_rotvec()

    state = np.concatenate([
        obs["robot0_eef_pos"],
        rotation_vector,
        obs["robot0_gripper_qpos"],
    ]).astype(np.float32)

    def prepare_image(raw_image):
        # Match notebook 06: upright RGB.
        image = np.ascontiguousarray(raw_image[::-1])

        # HWC uint8 → CHW float32 in [0, 1].
        return torch.from_numpy(image).permute(2, 0, 1).float()/ 255.0


    return {
        "observation.images.image": prepare_image(obs["agentview_image"]),
        "observation.images.image2": prepare_image(obs["robot0_eye_in_hand_image"]),
        "observation.state": torch.from_numpy(state),
        "task": instruction,
    }


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

from libero.libero import benchmark
suite = benchmark.get_benchmark_dict()['libero_spatial']()
task_name = "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate"

task_id = next(i for i in range(suite.n_tasks) if suite.get_task(i).name == task_name)
task = suite.get_task(task_id)

task_id, task.language
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
MAX_STEPS=6000  # Demonstrations take ~4,500 steps; 500 steps would be too short.
SEED=42
np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
if 'env' in globals(): env.close()
bddl_path=os.path.join(get_libero_path('bddl_files'),task.problem_folder,task.bddl_file)
print('Creating evaluation environment...',flush=True)
env=OffScreenRenderEnv(bddl_file_name=bddl_path,camera_heights=256,camera_widths=256,
                       horizon=MAX_STEPS+1,ignore_done=True)
env.seed(SEED)


In [ ]:
def reset_to_recorded_start():
    """Restore scene XML and state; check the pre-action state matches frame zero."""
    with h5py.File(source_episode, "r") as f:
        initial_state = f["initial_sim_state"][:]
        model_xml = f["model_xml"][()]
        first_state = f["state"][0]
    if isinstance(model_xml, bytes):
        model_xml = model_xml.decode()
    env.reset()
    env.reset_from_xml_string(model_xml)
    restored_obs = env.set_init_state(initial_state)
    np.testing.assert_allclose(
        np.asarray(env.get_sim_state()), initial_state, atol=1e-6, rtol=0,
        err_msg="Simulator did not restore the recorded state",
    )
    live_state = np.r_[restored_obs["robot0_eef_pos"],
                       restored_obs["robot0_eef_quat"],
                       restored_obs["robot0_gripper_qpos"]]
    np.testing.assert_allclose(live_state, first_state, atol=1e-5, rtol=0)
    policy.reset()
    preprocess.reset()
    postprocess.reset()
    print("Evaluation reset: recorded training scene; state checks passed.")
    return restored_obs

obs = reset_to_recorded_start()


In [ ]:
with torch.inference_mode():
    policy_observation = make_policy_observation(obs, task.language)
    model_batch = preprocess(policy_observation)

    predicted_action = policy.select_action(model_batch)
    action_tensor = postprocess(predicted_action)

action = action_tensor.squeeze(0).detach().cpu().numpy()

print("Action shape:", action.shape)
print("Action:", action)

assert action.shape == (7,)
assert np.isfinite(action).all()

# Clear the queued actions from this diagnostic prediction.
policy.reset()
preprocess.reset()
postprocess.reset()

## Run policy evaluation
Continue past the first task-success signal to allow policy-controlled release. Stop after 20 consecutive steps with task success, bowl–plate contact, no finger–bowl contact, and a negative gripper command. This is an automatic release heuristic and still needs replay review. Raw predictions are clipped only to environment action bounds; no speed gain or gripper threshold is applied.

In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display
obs=reset_to_recorded_start()
policy.eval()
sim=env.env.sim
gripper=env.env.robots[0].gripper
bowl_geoms={i for i in range(sim.model.ngeom) if (sim.model.geom_id2name(i) or '').startswith('akita_black_bowl_1_')}
plate_geoms={i for i in range(sim.model.ngeom) if (sim.model.geom_id2name(i) or '').startswith('plate_1_')}
finger_geoms={sim.model.geom_name2id(name) for names in gripper.important_geoms.values() for name in names}
assert bowl_geoms and plate_geoms and finger_geoms

def touching(a,b):
    return any(c.dist<=0.0001 and ((c.geom1 in a and c.geom2 in b) or (c.geom2 in a and c.geom1 in b))
               for c in (sim.data.contact[i] for i in range(sim.data.ncon)))

eval_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
eval_dir=Path('/content/humanoid_batch_eval')/eval_id
eval_dir.mkdir(parents=True,exist_ok=False)
video_path=eval_dir/'rollout.mp4'
video=imageio.get_writer(str(video_path),fps=20)
low,high=env.env.action_spec
rollout_log=[]
release_streak=0
initial_eef=obs['robot0_eef_pos'].copy()
started=time.monotonic()
report=dict(checkpoint=str(checkpoint_dir),training_run=training_run,checkpoint_step=evaluated_step,
            source_episode=str(source_episode),seed=SEED,max_steps=MAX_STEPS,
            reset_mode='recorded model XML and simulator state',evaluation_scope='training scene',
            task_success_ever=False,release_check_passed=False,visual_review_required=True,
            status='running',clipped_steps=0)
(eval_dir/'evaluation_report.json').write_text(json.dumps(report,indent=2))
try:
    with torch.inference_mode():
        for step_index in range(MAX_STEPS):
            batch_input=preprocess(make_policy_observation(obs,task.language))
            action=postprocess(policy.select_action(batch_input)).squeeze(0).detach().cpu().numpy()
            assert action.shape==(7,) and np.isfinite(action).all()
            bounded=np.clip(action,low,high)
            report['clipped_steps']+=int(np.any(action!=bounded))
            obs,reward,done,info=env.step(bounded)
            success=bool(env.check_success())
            finger_contact=touching(finger_geoms,bowl_geoms)
            plate_contact=touching(bowl_geoms,plate_geoms)
            released=success and plate_contact and not finger_contact and bounded[-1]<0
            release_streak=release_streak+1 if released else 0
            report['task_success_ever'] |= success
            if success and 'first_success_step' not in report: report['first_success_step']=step_index+1
            rollout_log.append(dict(step=step_index,action=bounded.tolist(),prediction=action.tolist(),
                                    eef=obs['robot0_eef_pos'].tolist(),success=success,
                                    finger_bowl_contact=finger_contact,bowl_plate_contact=plate_contact))
            video.append_data(np.concatenate([obs['agentview_image'][::-1],obs['robot0_eye_in_hand_image'][::-1]],axis=1))
            if (step_index+1)%100==0:
                print(f'Step {step_index+1}/{MAX_STEPS} | success={success} | elapsed={(time.monotonic()-started)/60:.1f} min',flush=True)
            if release_streak>=20:
                report.update(status='passed_release_check',release_check_passed=True);break
            if bool(getattr(env.env,'done',False)) or (done and not success):
                report['status']='environment_terminated';break
        else: report['status']='step_limit'
except KeyboardInterrupt:
    report['status']='interrupted'
except Exception as exc:
    report.update(status='error',error=f'{type(exc).__name__}: {exc}')
finally:
    video.close()
    report.update(steps=len(rollout_log),simulated_seconds=len(rollout_log)/20,
                  wall_seconds=time.monotonic()-started,
                  final_task_success=bool(env.check_success()),
                  eef_displacement_m=(obs['robot0_eef_pos']-initial_eef).tolist())
    (eval_dir/'evaluation_report.json').write_text(json.dumps(report,indent=2))
    (eval_dir/'actions.json').write_text(json.dumps(rollout_log))
    env.close()
    drive_eval=project/'outputs/smolvla_batch_eval'/eval_id
    shutil.copytree(eval_dir,drive_eval)
    print(json.dumps(report,indent=2))
    print('Saved to Drive:',drive_eval)
if rollout_log: display(Video(str(video_path),embed=True))


Review whether the policy approaches, grasps, transfers and releases. Keep failed results too. Report this as one trial, not a success rate. After this first trial, repeat with documented policy random seeds if time allows. If it fails, the action log and video help decide whether more training or a pipeline correction is needed.

# Offline prediction checks — no simulation or training
Run only this section after installing the notebook dependencies. It can run after a runtime restart. Checks the saved 2,000-step checkpoint on recorded observations from three episodes. CPU is supported but inference can be slow; reduce EPISODE_INDICES or REPEATS if needed. No optimizer updates occur. Preprocessing checks run before model loading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json, zipfile, shutil, gc
import numpy as np
import pandas as pd
import torch
import h5py
from lerobot.datasets.lerobot_dataset import LeRobotDataset

project=Path('/content/drive/MyDrive/Projects/humanoid-challenge')
export_id='20261015T103228_810691Z'
training_run='20261015T130807_429731Z'
dataset_path=Path('/content/humanoid_dataset_exports')/export_id/'lerobot_dataset'
checkpoint_dir=Path('/content/humanoid_training')/training_run/'step_002000'

for target, archive in [(dataset_path,project/'data/lerobot_exports'/export_id/'lerobot_dataset.zip'),
                        (checkpoint_dir,project/'outputs/smolvla_batch'/training_run/'step_002000.zip')]:
    marker=target/('meta/info.json' if target==dataset_path else 'config.json')

    if not marker.exists():
        target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            z.extractall(target)

conversion=json.loads((checkpoint_dir/'dataset_conversion_manifest.json').read_text())
recorded_dataset=LeRobotDataset(repo_id='local/humanoid_human_batch',root=dataset_path)
EPISODE_INDICES=[0,7,15]
REPEATS=3
PHASES=['orient','above_rim','approach','insert']
check_dir=Path('/content/humanoid_offline_checks')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
check_dir.mkdir(parents=True,exist_ok=False)


In [ ]:
import numpy as np
import torch
from scipy.spatial.transform import Rotation

def make_policy_observation(obs, instruction):
    # Quaternion from LIBERO uses XYZW ordering.
    rotation_vector = Rotation.from_quat(obs["robot0_eef_quat"]).as_rotvec()

    state = np.concatenate([
        obs["robot0_eef_pos"],
        rotation_vector,
        obs["robot0_gripper_qpos"],
    ]).astype(np.float32)

    def prepare_image(raw_image):
        # Match notebook 06: upright RGB.
        image = np.ascontiguousarray(raw_image[::-1])

        # HWC uint8 → CHW float32 in [0, 1].
        return torch.from_numpy(image).permute(2, 0, 1).float()/ 255.0


    return {
        "observation.images.image": prepare_image(obs["agentview_image"]),
        "observation.images.image2": prepare_image(obs["robot0_eye_in_hand_image"]),
        "observation.state": torch.from_numpy(state),
        "task": instruction,
    }


In [ ]:
# Construct evaluation observations from the SAME recorded samples.
# Undo the recorder's vertical flip, then call the real evaluation conversion.
# This verifies this conversion; it does not prove fresh simulator rendering is identical.
samples=[]

for episode_index in EPISODE_INDICES:
    source=conversion['sources'][episode_index]
    local_source=check_dir/'source.h5'
    shutil.copy2(project/source['source_h5'],local_source)

    with h5py.File(local_source,'r') as f:
        phases=f['phase'].asstr()[:]

        for phase in PHASES:
            indices=np.flatnonzero(phases==phase)

            if not len(indices):
                raise ValueError(f'Missing phase {phase} in {source["demo"]}')

            index=int(indices[len(indices)//2])
            global_index=source['dataset_from_index']+index
            item=recorded_dataset[global_index]
            state=f['state'][index]

            raw={'robot0_eef_pos':state[:3], 'robot0_eef_quat':state[3:7],
                 'robot0_gripper_qpos':state[7:9],
                 'agentview_image':f['agentview_image'][index][::-1].copy(),
                 'robot0_eye_in_hand_image':f['wrist_image'][index][::-1].copy()}

            evaluation_input=make_policy_observation(raw,item['task'])
            training_input={key:item[key] for key in evaluation_input}

            for key in evaluation_input:
                if key=='task':
                    assert evaluation_input[key]==training_input[key]
                else:
                    torch.testing.assert_close(evaluation_input[key],training_input[key],atol=1e-6,rtol=0)

            np.testing.assert_allclose(item['action'].numpy(),f['action'][index],atol=1e-7)
            samples.append(dict(demo=source['demo'],episode=episode_index,phase=phase,
                                local_frame=index,global_frame=global_index,
                                training_input=training_input,evaluation_input=evaluation_input,
                                target=f['action'][index].copy()))
    local_source.unlink()
print('Raw input parity and action alignment passed for',len(samples),'samples.')


In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.factory import make_pre_post_processors
# Free any previous training model/optimizer; this is evaluation only.
optimizer=None
trainable_parameters=[]
if 'policy' in globals():del policy
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
device='cuda' if torch.cuda.is_available() else 'cpu'
print('Inference device:',device,flush=True)
config=SmolVLAConfig.from_pretrained(checkpoint_dir)
config.device=device
policy=SmolVLAPolicy.from_pretrained(checkpoint_dir,config=config).to(device).eval()
preprocess,postprocess=make_pre_post_processors(policy.config,pretrained_path=checkpoint_dir,
    preprocessor_overrides={'device_processor':{'device':device}})
assert policy.config.output_features['action'].shape==(7,)


In [ ]:
def compare_tree(a,b,path='batch'):
    if isinstance(a,torch.Tensor):
        torch.testing.assert_close(a,b,atol=1e-6,rtol=0,msg=path)
    elif isinstance(a,dict):
        assert a.keys()==b.keys(),path
        for k in a:compare_tree(a[k],b[k],path+'.'+k)
    elif isinstance(a,(list,tuple)):
        assert len(a)==len(b),path
        for i,(x,y) in enumerate(zip(a,b)):compare_tree(x,y,path+str(i))
    else:assert a==b,path

rows=[]
for sample in samples:
    preprocess.reset()
    a=preprocess(dict(sample['training_input']))
    preprocess.reset()
    b=preprocess(dict(sample['evaluation_input']))
    compare_tree(a,b)
    predictions=[]
    for repeat in range(REPEATS):
        seed=100+repeat
        torch.manual_seed(seed)
        if device=='cuda':torch.cuda.manual_seed_all(seed)
        policy.reset();preprocess.reset();postprocess.reset()
        with torch.inference_mode():
            batch_input=preprocess(dict(sample['evaluation_input']))
            pred=postprocess(policy.select_action(batch_input)).squeeze(0).detach().cpu().numpy()
        assert pred.shape==(7,) and np.isfinite(pred).all()
        predictions.append(pred)
    predictions=np.stack(predictions)
    target=sample['target']
    mean=predictions.mean(axis=0)
    error=np.abs(predictions-target)
    row={k:sample[k] for k in ['demo','episode','phase','local_frame','global_frame']}
    row.update(translation_mae=float(error[:,:3].mean()),rotation_mae=float(error[:,3:6].mean()),
               gripper_mae=float(error[:,6].mean()),
               translation_std=float(predictions[:,:3].std(axis=0).mean()))
    for dim,label in enumerate(['dx','dy','dz','rx','ry','rz','gripper']):
        row['target_'+label]=float(target[dim]);row['prediction_'+label]=float(mean[dim])
    row['predictions']=predictions.tolist()
    rows.append(row)
    print(sample['demo'],sample['phase'],'frame',sample['local_frame'],
          '\n target:',np.round(target,4),'\n prediction mean:',np.round(mean,4),flush=True)
results=pd.DataFrame(rows)
results.to_csv(check_dir/'prediction_checks.csv',index=False)
(check_dir/'prediction_checks.json').write_text(json.dumps(rows,indent=2))
print('Saved-processor parity also passed.')
display(results[['demo','phase','local_frame','translation_mae','rotation_mae','gripper_mae','translation_std']])


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(3,1,figsize=(13,9),sharex=True)
labels=[r['demo']+' / '+r['phase'] for r in rows]
x=np.arange(len(rows))

for axis,dimension in zip(axes,['dx','dy','dz']):
    axis.plot(x,results['target_'+dimension],'o-',label='Recorded target')
    axis.plot(x,results['prediction_'+dimension],'x--',label='Mean prediction')
    std=[np.std(np.array(r['predictions'])[:,['dx','dy','dz'].index(dimension)]) for r in rows]
    axis.errorbar(x,results['prediction_'+dimension],yerr=std,fmt='none',alpha=.5)
    axis.axhline(0,color='gray',linewidth=.5);axis.set_ylabel(dimension+' (command units)')
    axis.legend();axis.grid(alpha=.2)
axes[-1].set_xticks(x,labels,rotation=60,ha='right')
fig.tight_layout();fig.savefig(check_dir/'translation_predictions.png',dpi=150);plt.show()
(check_dir/'check_metadata.json').write_text(json.dumps(dict(checkpoint=str(checkpoint_dir),
    repeats=REPEATS,seeds=list(range(100,100+REPEATS)),episodes=EPISODE_INDICES,
    input_parity_verified=True,scope='First predicted action on recorded observations; not closed-loop evaluation'),indent=2))
destination=project/'outputs/smolvla_offline_checks'/check_dir.name
shutil.copytree(check_dir,destination)
print('Saved diagnostics:',destination)


## Startup diagnostic: frames 0–350
Run after the offline-check setup and checkpoint-loading cells. Reuses the saved policy without training or simulation. Defaults to 36 observations × 3 predictions = 108 inference calls; CPU can take time. Uses recorded observations, so this isolates prediction errors on the demonstrated trajectory rather than measuring closed-loop divergence.


In [ ]:
# Reuses the model, saved processors, recorded_dataset and conversion from offline checks.
# No simulation, training, or checkpoint reload.
import time
import pandas as pd
import matplotlib.pyplot as plt

STARTUP_STRIDE = 10  # Set to 1 only if you need every frame; much slower on CPU.
STARTUP_REPEATS = 3

source = conversion['sources'][0]
frame_indices = list(range(0, min(351, source['frames']), STARTUP_STRIDE))
startup_dir = Path('/content/humanoid_startup_checks') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
startup_dir.mkdir(parents=True, exist_ok=False)

records = []
started = time.monotonic()
policy.eval()
labels = ['dx','dy','dz','rx','ry','rz','gripper']

for frame in frame_indices:
    item = recorded_dataset[source['dataset_from_index'] + frame]
    observation = {key:item[key] for key in ['observation.images.image','observation.images.image2','observation.state','task']}
    target = item['action'].numpy()
    predictions = []

    for repeat in range(STARTUP_REPEATS):
        torch.manual_seed(100 + repeat)
        if device == 'cuda':
            torch.cuda.manual_seed_all(100 + repeat)

        policy.reset(); preprocess.reset(); postprocess.reset()

        with torch.inference_mode():
            prediction = postprocess(policy.select_action(preprocess(dict(observation))))
        prediction = prediction.squeeze(0).detach().cpu().numpy()
        assert prediction.shape == (7,) and np.isfinite(prediction).all()
        predictions.append(prediction)

    predictions = np.stack(predictions)
    row = dict(frame=frame, seconds=frame/20, predictions=predictions.tolist())

    for d,label in enumerate(labels):
        row['target_'+label] = float(target[d])
        row['mean_'+label] = float(predictions[:,d].mean())
        row['std_'+label] = float(predictions[:,d].std())
    records.append(row)

    # Persist partial progress locally in case CPU execution is interrupted.
    (startup_dir/'predictions.json').write_text(json.dumps(records,indent=2))
    print(f'Frame {frame}/350 | {len(records)}/{len(frame_indices)} samples | elapsed {(time.monotonic()-started)/60:.1f} min',flush=True)

startup = pd.DataFrame(records)
startup.to_csv(startup_dir/'predictions.csv',index=False)

fig, axes = plt.subplots(7,1,figsize=(12,17),sharex=True)

for ax,label in zip(axes,labels):
    ax.plot(startup['frame'],startup['target_'+label],label='Recorded target',color='black')
    ax.plot(startup['frame'],startup['mean_'+label],label='Mean prediction',color='tab:blue')
    ax.fill_between(startup['frame'],startup['mean_'+label]-startup['std_'+label],
                    startup['mean_'+label]+startup['std_'+label],alpha=.2,color='tab:blue',label='±1 sample std')
    ax.axhline(0,color='gray',linewidth=.5)
    ax.set_ylabel(label); ax.grid(alpha=.2)

axes[0].legend(ncol=3)
axes[-1].set_xlabel('Recorded episode frame (20 Hz)')

fig.suptitle('Startup: first predicted action at recorded observations',y=1.0)
fig.tight_layout();fig.savefig(startup_dir/'startup_predictions.png',dpi=160);plt.show()

summary=[]
for label in labels:
    error=startup['mean_'+label]-startup['target_'+label]
    summary.append(dict(axis=label,mean_bias=float(error.mean()),
                        mean_absolute_error=float(error.abs().mean()),
                        mean_prediction_std=float(startup['std_'+label].mean())))
display(pd.DataFrame(summary))

(startup_dir/'metadata.json').write_text(json.dumps(dict(checkpoint=str(checkpoint_dir),
    demo=source['demo'],frames=frame_indices,seeds=list(range(100,100+STARTUP_REPEATS)),
    note='Recorded observations, first action only. Pointwise errors are not integrated robot displacements; std is across three random predictions.'),indent=2))
destination=project/'outputs/smolvla_startup_checks'/startup_dir.name
shutil.copytree(startup_dir,destination)
print('Saved:',destination)


# Continue training from a chosen checkpoint
Run ONLY this section to resume, after installing dependencies. Do not rerun the fresh-model training cells above. Requires GPU. Restores model, saved processors, Adam optimizer state, step count and random states. Sampling starts a new iterator, so this is continuation, not exact replay of an interrupted minibatch sequence.

Optional startup emphasis samples a chosen fraction of chunks whose starting frame lies within 0–350. All remaining frames retain sampling probability. This changes sampling, not target actions, loss definition, or normalization. It may help startup learning but cannot remove ambiguity in a scripted stationary wait.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, zipfile, shutil, hashlib, random, gc, time
from datetime import datetime, timezone
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors
project=Path('/content/drive/MyDrive/Projects/humanoid-challenge')

# EDIT THESE SETTINGS. Only load your own trusted training archives.
RESUME_RUN = '20261015T130807_429731Z'
RESUME_STEP = 2000
ADDITIONAL_STEPS = 2000
SAVE_EVERY = 500
STARTUP_LAST_FRAME = 350
STARTUP_FRACTION = 0.30  # 30% of sampled chunks start in startup; None = uniform sampling.
LR_OVERRIDE = None       # None keeps the learning rate stored in the optimizer.
BATCH_SIZE = 1
assert torch.cuda.is_available(), 'Continuation needs a GPU runtime.'
assert ADDITIONAL_STEPS>0 and SAVE_EVERY>0
assert STARTUP_FRACTION is None or 0<STARTUP_FRACTION<1
resume_dir=Path('/content/humanoid_training')/RESUME_RUN/f'step_{RESUME_STEP:06d}'
resume_archive=project/'outputs/smolvla_batch'/RESUME_RUN/f'step_{RESUME_STEP:06d}.zip'
if not (resume_dir/'training_state.pt').exists():
    resume_dir.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(resume_archive) as z:z.extractall(resume_dir)
parent_log=json.loads((resume_dir/'training_log.json').read_text())
assert parent_log['completed_steps']==RESUME_STEP
SOURCE_RUN=parent_log['dataset_export']
REPO_ID=parent_log['repo_id']
MODEL_ID=parent_log['base_model']
SEED=parent_log['seed']
dataset_path=Path('/content/humanoid_dataset_exports')/SOURCE_RUN/'lerobot_dataset'
if not (dataset_path/'conversion_manifest.json').exists():
    dataset_path.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(project/'data/lerobot_exports'/SOURCE_RUN/'lerobot_dataset.zip') as z:z.extractall(dataset_path)
conversion=json.loads((dataset_path/'conversion_manifest.json').read_text())
assert conversion==json.loads((resume_dir/'dataset_conversion_manifest.json').read_text()), 'Dataset differs from checkpoint.'


In [ ]:
optimizer=None
trainable_parameters=[]
if 'policy' in globals():del policy
gc.collect();torch.cuda.empty_cache()
config=SmolVLAConfig.from_pretrained(resume_dir)
config.device='cuda'
policy=SmolVLAPolicy.from_pretrained(resume_dir,config=config).to('cuda')
preprocess,postprocess=make_pre_post_processors(policy.config,pretrained_path=resume_dir,
    preprocessor_overrides={'device_processor':{'device':'cuda'}})
trainable_parameters=[p for p in policy.parameters() if p.requires_grad]
optimizer=torch.optim.AdamW(trainable_parameters,lr=parent_log['learning_rate'],weight_decay=1e-2)
# weights_only=False is needed for saved NumPy/Python RNG state. Trusted own checkpoint only.
saved_training=torch.load(resume_dir/'training_state.pt',map_location='cpu',weights_only=False)
assert saved_training['step']==RESUME_STEP
optimizer.load_state_dict(saved_training['optimizer'])
if LR_OVERRIDE is not None:
    for group in optimizer.param_groups:group['lr']=LR_OVERRIDE
LEARNING_RATE=optimizer.param_groups[0]['lr']
random.setstate(saved_training['python_rng'])
np.random.set_state(saved_training['numpy_rng'])
torch.set_rng_state(saved_training['torch_rng'])
if len(saved_training['cuda_rng'])==torch.cuda.device_count():
    torch.cuda.set_rng_state_all(saved_training['cuda_rng'])
else:
    torch.cuda.manual_seed_all(SEED)
    print('GPU count changed: CUDA RNG reseeded.')
del saved_training
training_dataset=LeRobotDataset(repo_id=REPO_ID,root=dataset_path,
    delta_timestamps={'action':[i/conversion['fps'] for i in range(policy.config.chunk_size)]})
assert len(training_dataset)==conversion['frames']
startup_mask=torch.zeros(len(training_dataset),dtype=torch.bool)
for source in conversion['sources']:
    start=source['dataset_from_index']
    stop=min(source['dataset_to_index'],start+STARTUP_LAST_FRAME+1)
    startup_mask[start:stop]=True
natural_fraction=float(startup_mask.float().mean())
if STARTUP_FRACTION is None:
    loader=DataLoader(training_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
else:
    assert 0<natural_fraction<1
    weights=torch.where(startup_mask,
        torch.tensor(STARTUP_FRACTION/int(startup_mask.sum()),dtype=torch.double),
        torch.tensor((1-STARTUP_FRACTION)/int((~startup_mask).sum()),dtype=torch.double))
    sampler=WeightedRandomSampler(weights,num_samples=len(training_dataset),replacement=True)
    loader=DataLoader(training_dataset,batch_size=BATCH_SIZE,sampler=sampler,num_workers=0)
print(f'Startup share: natural={natural_fraction:.1%}, requested={STARTUP_FRACTION if STARTUP_FRACTION is not None else natural_fraction:.1%}')
print('Resuming step',RESUME_STEP,'to',RESUME_STEP+ADDITIONAL_STEPS,'with learning rate',LEARNING_RATE)
training_details=json.loads((resume_dir/'training_details.json').read_text())
assert len(training_details)==RESUME_STEP
completed_steps=RESUME_STEP
TOTAL_STEPS=RESUME_STEP+ADDITIONAL_STEPS
last_saved_step=RESUME_STEP
run_id=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
local_run=Path('/content/humanoid_training')/run_id
local_run.mkdir(parents=True,exist_ok=False)
drive_run=project/'outputs/smolvla_batch'/run_id
drive_run.mkdir(parents=True,exist_ok=False)


In [ ]:
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def save_checkpoint(step):
    global last_saved_step, checkpoint_dir
    checkpoint_dir=local_run/f'step_{step:06d}'
    checkpoint_dir.mkdir(parents=True,exist_ok=False)
    policy.save_pretrained(checkpoint_dir)
    preprocess.save_pretrained(checkpoint_dir,config_filename='policy_preprocessor.json')
    postprocess.save_pretrained(checkpoint_dir,config_filename='policy_postprocessor.json')
    torch.save({'step':step,'optimizer':optimizer.state_dict(),
                'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),
                'numpy_rng':np.random.get_state(),'python_rng':random.getstate()},
               checkpoint_dir/'training_state.pt')
    # RNG state alone does not preserve an in-flight DataLoader iterator's order.
    log=dict(completed_steps=step,batch_size=loader.batch_size,learning_rate=LEARNING_RATE,
             parent_checkpoint=str(resume_dir),
             startup_fraction=STARTUP_FRACTION,startup_last_frame=STARTUP_LAST_FRAME,
             continuation_start_step=RESUME_STEP,
             seed=SEED,base_model=MODEL_ID,repo_id=REPO_ID,dataset_export=SOURCE_RUN,
             dataset_episodes=16,dataset_frames=71274,
             loss_history=[x['loss'] for x in training_details],
             purpose='fixed-scene multi-demonstration proof of concept; no held-out split')
    (checkpoint_dir/'training_log.json').write_text(json.dumps(log,indent=2))
    (checkpoint_dir/'training_details.json').write_text(json.dumps(training_details,indent=2))
    shutil.copy2(dataset_path/'conversion_manifest.json',checkpoint_dir/'dataset_conversion_manifest.json')
    archive=Path(shutil.make_archive(str(checkpoint_dir),'zip',root_dir=checkpoint_dir))
    destination=drive_run/archive.name
    temporary=destination.with_suffix('.zip.partial')
    print('Saving checkpoint to Drive:',destination,flush=True)
    shutil.copy2(archive,temporary)
    assert digest(temporary)==digest(archive), 'Checkpoint copy checksum mismatch'
    temporary.replace(destination)
    (drive_run/'latest.json').write_text(json.dumps({'step':step,'archive':destination.name,'sha256':digest(archive)},indent=2))
    last_saved_step=step
    print('Verified checkpoint:',destination,flush=True)


In [ ]:
assert completed_steps == RESUME_STEP, 'Already executed: restore a saved checkpoint before another continuation.'
policy.train()
policy.reset()
iterator=iter(loader)
started=time.monotonic()
try:
    for step in range(RESUME_STEP+1,TOTAL_STEPS+1):
        try: raw_batch=next(iterator)
        except StopIteration:
            iterator=iter(loader)
            raw_batch=next(iterator)
        model_batch=preprocess(raw_batch)
        optimizer.zero_grad(set_to_none=True)
        loss,metrics=policy.forward(model_batch)
        assert torch.isfinite(loss).item(), 'Nonfinite training loss'
        loss.backward()
        norm=torch.nn.utils.clip_grad_norm_(trainable_parameters,1.0,error_if_nonfinite=True)
        optimizer.step()
        completed_steps=step
        training_details.append(dict(step=step,loss=float(loss.detach().cpu()),
            gradient_norm_before_clipping=float(norm),
            frame_indices=raw_batch['index'].tolist(),
            episode_indices=raw_batch['episode_index'].tolist()))
        if step%20==0:
            recent=np.mean([x['loss'] for x in training_details[-20:]])
            print(f'Step {step}/{TOTAL_STEPS} | mean loss {recent:.4f} | elapsed {(time.monotonic()-started)/60:.1f} min',flush=True)
        if step%SAVE_EVERY==0: save_checkpoint(step)

except KeyboardInterrupt:
    print('Interrupted. Saving the last completed update if needed.',flush=True)
    if completed_steps and last_saved_step!=completed_steps:
        save_checkpoint(completed_steps)
    raise
if completed_steps and last_saved_step!=completed_steps:
    save_checkpoint(completed_steps)
print('Local checkpoint:',checkpoint_dir)
print('Drive checkpoints:',drive_run)


After this continuation, rerun the recorded startup prediction checks on the NEW checkpoint before another long rollout. The older diagnostic setup cells contain fixed checkpoint paths: update them to the new printed run ID and cumulative step (default 4,000). Checkpoints are saved in a new run directory; the original run remains intact.